# Corrected batch metrics computation

This version transforms each ego–neighbour pair from exiD global coordinates into one common ego-aligned longitudinal/lateral frame before computing PCAD. Longitudinal and lateral jerk are computed from `lonAcceleration` and `latAcceleration`. Compound neighbour IDs such as `287;285` are preserved and evaluated individually.

All outputs from the previous coordinate implementation have been cleared. Rerun the code to regenerate the summary and raw-PCAD CSV files.

The non-reference threshold outputs `p95_pcad`, `pcad_threshold_used`, and `high_pcad_duration_ratio` have been removed together with their related settings and calculations.

Output files are now organized automatically:
- Summary CSV files: `results/summary/`
- Raw PCAD CSV files: `results/raw/`


In [ ]:
import os
import glob
import ast
import time
import numpy as np
import pandas as pd

from workflow.pcad_python import pcad_function, PCADParams


# ============================================================
# Settings
# ============================================================
base_path = os.path.join("data")

merging_summary_path = os.path.join("results", "summary", "exid_merging_vehicle_summary.csv")

params = PCADParams()

# Test one recording first.
# Example: 0, 72, 83
# Set to None when processing all 93 recordings.
test_recording_id = None

# Test only first N merging vehicles in the selected recording.
# Set to None if you want all merging vehicles in that recording.
test_num_vehicles = None

# Save frame-level raw PCAD data for evolution plots
save_raw_pcad = True

# Keep detailed PCAD DataFrames in memory for debugging.
# For full batch, keep False to save memory.
save_debug_pcad = False

# Stop immediately if any error occurs
stop_on_error = True


# Do not stop just because all PCAD values are zero.
# We are testing whether all-zero PCAD can naturally happen.
raise_if_valid_pairs_but_all_zero = False

# Output folders
summary_output_dir = os.path.join("results", "summary")
raw_output_dir = os.path.join("results", "raw")

# Create folders automatically if they do not exist
os.makedirs(summary_output_dir, exist_ok=True)
os.makedirs(raw_output_dir, exist_ok=True)

# Output paths
if test_recording_id is None:
    output_path = os.path.join(
        summary_output_dir,
        "exid_metrics_summary_all_recordings.csv"
    )
    raw_pcad_output_path = os.path.join(
        raw_output_dir,
        "exid_all_merging_vehicle_pcad_raw.csv"
    )
else:
    if test_num_vehicles is None:
        output_path = os.path.join(
            summary_output_dir,
            f"exid_metrics_summary_recording_{test_recording_id:02d}.csv"
        )
        raw_pcad_output_path = os.path.join(
            raw_output_dir,
            f"exid_pcad_raw_recording_{test_recording_id:02d}.csv"
        )
    else:
        output_path = os.path.join(
            summary_output_dir,
            f"exid_metrics_summary_recording_{test_recording_id:02d}_first{test_num_vehicles}.csv"
        )
        raw_pcad_output_path = os.path.join(
            raw_output_dir,
            f"exid_pcad_raw_recording_{test_recording_id:02d}_first{test_num_vehicles}.csv"
        )


# ============================================================
# Required columns
# ============================================================
surrounding_cols = [
    "leadId",
    "rearId",
    "leftLeadId",
    "leftRearId",
    "leftAlongsideId",
    "rightLeadId",
    "rightRearId",
    "rightAlongsideId"
]

state_cols = [
    "trackId",
    "frame",
    "xCenter",
    "yCenter",
    "heading",
    "xVelocity",
    "yVelocity",
    "xAcceleration",
    "yAcceleration"
]

extra_metric_cols = [
    "lonVelocity",
    "lonAcceleration",
    "latAcceleration"
]

required_cols = list(dict.fromkeys(
    state_cols + surrounding_cols + extra_metric_cols
))


# ============================================================
# Find all exiD files
# ============================================================
tracks_files = sorted(glob.glob(os.path.join(base_path, "*_tracks.csv")))
tracks_meta_files = sorted(glob.glob(os.path.join(base_path, "*_tracksMeta.csv")))
recording_meta_files = sorted(glob.glob(os.path.join(base_path, "*_recordingMeta.csv")))

print(f"Found {len(tracks_files)} tracks files")
print(f"Found {len(tracks_meta_files)} tracksMeta files")
print(f"Found {len(recording_meta_files)} recordingMeta files")

if len(recording_meta_files) == 0:
    raise FileNotFoundError("No recordingMeta files found. Please check base_path.")


def get_recording_id_from_path(path):
    """
    Example:
    ../../data/04_tracks.csv -> 4
    """
    filename = os.path.basename(path)
    return int(filename.split("_")[0])


tracks_file_map = {
    get_recording_id_from_path(path): path
    for path in tracks_files
}

tracks_meta_file_map = {
    get_recording_id_from_path(path): path
    for path in tracks_meta_files
}

recording_meta_file_map = {
    get_recording_id_from_path(path): path
    for path in recording_meta_files
}


# ============================================================
# Load merging vehicle summary
# ============================================================
merging_summary_df = pd.read_csv(merging_summary_path)


def parse_merging_vehicle_ids(value):
    """
    Safely parse merging_vehicle_ids.
    Handles strings like "[39, 53]" and already-parsed lists.
    """
    if isinstance(value, list):
        return [int(v) for v in value]

    if isinstance(value, str):
        parsed = ast.literal_eval(value)
        return [int(v) for v in parsed]

    raise ValueError(f"Cannot parse merging_vehicle_ids value: {value}")


merging_summary_df["merging_vehicle_ids"] = (
    merging_summary_df["merging_vehicle_ids"]
    .apply(parse_merging_vehicle_ids)
)

print(f"Loaded merging summary: {len(merging_summary_df)} recordings")
print(
    "Total merging vehicles in full summary:",
    merging_summary_df["merging_vehicle_ids"].apply(len).sum()
)


# ============================================================
# Select test recording
# ============================================================
if test_recording_id is not None:
    merging_summary_df = merging_summary_df[
        merging_summary_df["recording_id"] == test_recording_id
    ].copy()

    if merging_summary_df.empty:
        raise ValueError(
            f"Recording {test_recording_id} not found in exid_merging_vehicle_summary."
        )


# ============================================================
# Select first N merging vehicles for testing
# ============================================================
if test_num_vehicles is not None:
    merging_summary_df["merging_vehicle_ids"] = (
        merging_summary_df["merging_vehicle_ids"]
        .apply(lambda ids: ids[:test_num_vehicles])
    )

    merging_summary_df["num_merging_vehicles"] = (
        merging_summary_df["merging_vehicle_ids"]
        .apply(len)
    )

    if merging_summary_df["num_merging_vehicles"].sum() == 0:
        raise ValueError(
            f"No merging vehicles selected for recording {test_recording_id}."
        )


print("\nSelected recordings:")
display(
    merging_summary_df[
        [
            "recording_id",
            "merging_lanelet_id",
            "num_merging_vehicles",
            "merging_vehicle_ids"
        ]
    ]
)

print(
    "Total selected merging vehicles:",
    merging_summary_df["merging_vehicle_ids"].apply(len).sum()
)


# ============================================================
# Data normalization
# ============================================================
def normalize_tracks_df_types(tracks_df, surrounding_cols):
    """
    Normalize columns after reading CSV.

    ``trackId`` and ``frame`` are stored as integers. Surrounding-vehicle
    columns are deliberately kept as object values because exiD alongside
    fields can contain more than one ID, for example ``"287;285"``.
    Motion and heading columns are converted to numeric floats.
    """

    tracks_df["trackId"] = (
        pd.to_numeric(tracks_df["trackId"], errors="coerce")
        .astype("int64")
    )

    tracks_df["frame"] = (
        pd.to_numeric(tracks_df["frame"], errors="coerce")
        .astype("int64")
    )

    # Keep compound neighbour IDs intact. They are parsed later by
    # parse_neighbor_ids().
    for col in surrounding_cols:
        tracks_df[col] = tracks_df[col].where(
            tracks_df[col].notna(),
            -1
        )

    numeric_cols = [
        "xCenter",
        "yCenter",
        "heading",
        "xVelocity",
        "yVelocity",
        "xAcceleration",
        "yAcceleration",
        "lonVelocity",
        "lonAcceleration",
        "latAcceleration"
    ]

    for col in numeric_cols:
        tracks_df[col] = pd.to_numeric(
            tracks_df[col],
            errors="coerce"
        )

    return tracks_df


# ============================================================
# State lookup and coordinate transformation
# ============================================================
def build_state_lookup(tracks_df):
    """
    Build a fast lookup table:
        (trackId, frame) -> GLOBAL exiD vehicle state

    Coordinate conversion is performed later for each ego-neighbour pair so
    both vehicles are expressed in one common ego-aligned frame.
    """

    lookup = {}

    cols = [
        "trackId",
        "frame",
        "xCenter",
        "yCenter",
        "heading",
        "xVelocity",
        "yVelocity",
        "xAcceleration",
        "yAcceleration"
    ]

    for row in tracks_df[cols].itertuples(index=False):
        track_id = int(row.trackId)
        frame = int(row.frame)

        lookup[(track_id, frame)] = {
            "trackId": track_id,
            "frame": frame,
            "x": float(row.xCenter),
            "y": float(row.yCenter),
            "heading_deg": float(row.heading),
            "vx": float(row.xVelocity),
            "vy": float(row.yVelocity),
            "ax": float(row.xAcceleration),
            "ay": float(row.yAcceleration),
        }

    return lookup


def parse_neighbor_ids(value):
    """
    Convert one exiD surrounding-vehicle field into valid track IDs.

    Most fields contain one ID or -1. Alongside fields can contain multiple
    IDs separated by semicolons, such as ``"287;285"``.
    """

    if value is None:
        return []

    if isinstance(value, str):
        text = value.strip()

        if text == "" or text.lower() == "nan":
            return []

        parts = (
            text.replace("[", "")
                .replace("]", "")
                .replace(",", ";")
                .split(";")
        )
    else:
        try:
            if pd.isna(value):
                return []
        except (TypeError, ValueError):
            pass

        if isinstance(value, (list, tuple, set, np.ndarray)):
            parts = list(value)
        else:
            parts = [value]

    ids = []

    for part in parts:
        try:
            track_id = int(float(str(part).strip()))
        except (TypeError, ValueError):
            continue

        if track_id != -1 and track_id not in ids:
            ids.append(track_id)

    return ids


def get_vehicle_state_at_frame_from_lookup(state_lookup, track_id, frame):
    """
    Return one vehicle's GLOBAL exiD state from the lookup table.
    """

    if track_id is None:
        return None

    try:
        if pd.isna(track_id):
            return None
    except (TypeError, ValueError):
        return None

    try:
        track_id = int(track_id)
    except (TypeError, ValueError):
        return None

    if track_id == -1:
        return None

    return state_lookup.get((track_id, int(frame)), None)


def rotate_global_vector_to_ego_frame(
    x_global,
    y_global,
    ego_heading_deg
):
    """
    Rotate a global 2-D vector into the ego-aligned coordinate system.

    Local x: ego forward / longitudinal direction.
    Local y: ego-left / lateral direction.

    The same ego heading must be used for the ego and its neighbour.
    """

    theta = np.deg2rad(float(ego_heading_deg))
    cos_theta = np.cos(theta)
    sin_theta = np.sin(theta)

    x_local = cos_theta * x_global + sin_theta * y_global
    y_local = -sin_theta * x_global + cos_theta * y_global

    return float(x_local), float(y_local)


def transform_pair_to_ego_frame(ego_state, neighbor_state):
    """
    Express one ego-neighbour pair in a common ego-aligned frame.

    The ego is translated to (0, 0). Relative position, both velocity vectors,
    and both acceleration vectors are then rotated using the ego heading.
    """

    if ego_state is None or neighbor_state is None:
        return None

    heading = ego_state["heading_deg"]

    dx_global = neighbor_state["x"] - ego_state["x"]
    dy_global = neighbor_state["y"] - ego_state["y"]

    x_neighbor_local, y_neighbor_local = (
        rotate_global_vector_to_ego_frame(
            dx_global,
            dy_global,
            heading
        )
    )

    vx_ego_local, vy_ego_local = rotate_global_vector_to_ego_frame(
        ego_state["vx"],
        ego_state["vy"],
        heading
    )

    vx_neighbor_local, vy_neighbor_local = (
        rotate_global_vector_to_ego_frame(
            neighbor_state["vx"],
            neighbor_state["vy"],
            heading
        )
    )

    ax_ego_local, ay_ego_local = rotate_global_vector_to_ego_frame(
        ego_state["ax"],
        ego_state["ay"],
        heading
    )

    ax_neighbor_local, ay_neighbor_local = (
        rotate_global_vector_to_ego_frame(
            neighbor_state["ax"],
            neighbor_state["ay"],
            heading
        )
    )

    return {
        "ego": {
            "x": 0.0,
            "y": 0.0,
            "vx": vx_ego_local,
            "vy": vy_ego_local,
            "ax": ax_ego_local,
            "ay": ay_ego_local,
        },
        "neighbor": {
            "x": x_neighbor_local,
            "y": y_neighbor_local,
            "vx": vx_neighbor_local,
            "vy": vy_neighbor_local,
            "ax": ax_neighbor_local,
            "ay": ay_neighbor_local,
        }
    }


def compute_pcad_between_states(
    ego_state,
    neighbor_state,
    params
):
    """
    Compute PCAD after converting global exiD states into a common,
    ego-aligned longitudinal/lateral frame.
    """

    if ego_state is None or neighbor_state is None:
        return np.nan

    local = transform_pair_to_ego_frame(
        ego_state,
        neighbor_state
    )

    ego = local["ego"]
    neighbor = local["neighbor"]

    try:
        return pcad_function(
            ego["x"], neighbor["x"],
            ego["y"], neighbor["y"],
            ego["vx"], neighbor["vx"],
            ego["vy"], neighbor["vy"],
            ego["ax"], neighbor["ax"],
            ego["ay"], neighbor["ay"],
            params=params
        )

    except Exception as e:
        raise RuntimeError(
            "pcad_function failed for one transformed ego-neighbor pair.\n"
            f"global ego_state = {ego_state}\n"
            f"global neighbor_state = {neighbor_state}\n"
            f"local pair = {local}"
        ) from e


def compute_pcad_for_frame(
    state_lookup,
    ego_row,
    ego_track_id,
    surrounding_cols,
    params
):
    """
    Compute frame-level PCAD in an ego-aligned coordinate frame.

    If an exiD surrounding field contains multiple IDs, the vehicle producing
    the highest valid PCAD is retained for that positional relation.
    """

    frame = int(ego_row["frame"])

    ego_state = get_vehicle_state_at_frame_from_lookup(
        state_lookup=state_lookup,
        track_id=ego_track_id,
        frame=frame
    )

    if ego_state is None:
        raise ValueError(
            f"Ego state not found for ego_track_id={ego_track_id}, "
            f"frame={frame}"
        )

    result = {"frame": frame}
    pcad_values = []

    for col in surrounding_cols:
        neighbor_ids = parse_neighbor_ids(ego_row[col])

        best_pcad = np.nan
        best_neighbor_id = -1

        for neighbor_id in neighbor_ids:
            neighbor_state = get_vehicle_state_at_frame_from_lookup(
                state_lookup=state_lookup,
                track_id=neighbor_id,
                frame=frame
            )

            pcad = compute_pcad_between_states(
                ego_state=ego_state,
                neighbor_state=neighbor_state,
                params=params
            )

            if pd.isna(pcad):
                continue

            if pd.isna(best_pcad) or pcad > best_pcad:
                best_pcad = float(pcad)
                best_neighbor_id = int(neighbor_id)

        result[f"{col}_trackId"] = best_neighbor_id
        result[f"pcad_{col}"] = best_pcad

        if not pd.isna(best_pcad):
            pcad_values.append(best_pcad)

    result["pcad_max"] = (
        float(np.nanmax(pcad_values))
        if len(pcad_values) > 0
        else np.nan
    )

    result["pcad_mean"] = (
        float(np.nanmean(pcad_values))
        if len(pcad_values) > 0
        else np.nan
    )

    return result


# ============================================================
# Comfort helper
# ============================================================
def compute_jerk_metrics(ego_df, frame_rate):
    """
    Compute direction-specific comfort metrics using exiD's vehicle-aligned
    longitudinal and lateral acceleration components.
    """

    if ego_df.empty or len(ego_df) < 2:
        return {
            "max_long_jerk": np.nan,
            "max_lat_jerk": np.nan,
            "rms_total_jerk": np.nan
        }

    required = [
        "frame",
        "lonAcceleration",
        "latAcceleration"
    ]

    missing = [
        col for col in required
        if col not in ego_df.columns
    ]

    if missing:
        raise ValueError(
            f"Cannot compute road-aligned jerk. "
            f"Missing columns: {missing}"
        )

    frames = ego_df["frame"].to_numpy(dtype=float)

    long_acc = ego_df[
        "lonAcceleration"
    ].to_numpy(dtype=float)

    lat_acc = ego_df[
        "latAcceleration"
    ].to_numpy(dtype=float)

    dt = np.diff(frames) / float(frame_rate)
    valid_dt = dt > 0

    long_diff = np.diff(long_acc)
    lat_diff = np.diff(lat_acc)

    long_jerk = np.full(
        len(long_acc),
        np.nan,
        dtype=float
    )

    lat_jerk = np.full(
        len(lat_acc),
        np.nan,
        dtype=float
    )

    valid_target_idx = np.flatnonzero(valid_dt) + 1

    long_jerk[valid_target_idx] = (
        long_diff[valid_dt] / dt[valid_dt]
    )

    lat_jerk[valid_target_idx] = (
        lat_diff[valid_dt] / dt[valid_dt]
    )

    total_jerk = np.sqrt(
        long_jerk ** 2 + lat_jerk ** 2
    )

    if np.all(np.isnan(long_jerk)):
        max_long_jerk = np.nan
    else:
        max_long_jerk = round(
            float(np.nanmax(np.abs(long_jerk))),
            3
        )

    if np.all(np.isnan(lat_jerk)):
        max_lat_jerk = np.nan
    else:
        max_lat_jerk = round(
            float(np.nanmax(np.abs(lat_jerk))),
            3
        )

    if np.all(np.isnan(total_jerk)):
        rms_total_jerk = np.nan
    else:
        rms_total_jerk = round(
            float(
                np.sqrt(
                    np.nanmean(total_jerk ** 2)
                )
            ),
            3
        )

    return {
        "max_long_jerk": max_long_jerk,
        "max_lat_jerk": max_lat_jerk,
        "rms_total_jerk": rms_total_jerk
    }


# ============================================================
# One ego vehicle metrics
# ============================================================
def compute_metrics_for_ego(
    tracks_df,
    state_lookup,
    recording_meta_df,
    recording_id,
    merging_lanelet_id,
    ego_track_id,
    params,
    return_pcad_df=False,
    raise_if_valid_pairs_but_all_zero=False
):
    """
    Compute one-row summary for one ego merging vehicle.

    Coordinate-safe implementation:
    - iterate ego frames;
    - retrieve each surrounding vehicle at the same frame;
    - transform the pair from global exiD coordinates into one common
      ego-aligned longitudinal/lateral frame;
    - call pcad_function() using the transformed states.
    """

    ego_df = (
        tracks_df[tracks_df["trackId"] == ego_track_id]
        .sort_values("frame")
        .copy()
        .reset_index(drop=True)
    )

    if ego_df.empty:
        raise ValueError(
            f"Ego vehicle {ego_track_id} not found in recording {recording_id:02d}."
        )

    frame_start = int(ego_df["frame"].min())
    frame_end = int(ego_df["frame"].max())
    num_frames = int(len(ego_df))

    if "frameRate" in recording_meta_df.columns:
        frame_rate = float(recording_meta_df.loc[0, "frameRate"])
    else:
        frame_rate = 25.0

    duration_s = num_frames / frame_rate

    pcad_rows = []

    for _, ego_row in ego_df.iterrows():
        try:
            pcad_rows.append(
                compute_pcad_for_frame(
                    state_lookup=state_lookup,
                    ego_row=ego_row,
                    ego_track_id=ego_track_id,
                    surrounding_cols=surrounding_cols,
                    params=params
                )
            )
        except Exception as e:
            raise RuntimeError(
                f"PCAD failed at recording {recording_id:02d}, "
                f"ego_track_id={ego_track_id}, "
                f"frame={int(ego_row['frame'])}"
            ) from e

    pcad_df = pd.DataFrame(pcad_rows)

    pcad_cols = [
        col for col in pcad_df.columns
        if col.startswith("pcad_")
        and col not in ["pcad_max", "pcad_mean"]
    ]

    if len(pcad_cols) == 0:
        raise ValueError(
            f"No pairwise PCAD columns created for recording {recording_id:02d}, "
            f"ego_track_id={ego_track_id}"
        )

    pcad_df["valid_pairwise_pcad_count"] = (
        pcad_df[pcad_cols]
        .notna()
        .sum(axis=1)
    )

    pcad_df["positive_pairwise_pcad_count"] = (
        pcad_df[pcad_cols]
        .gt(0)
        .sum(axis=1)
    )

    valid_pairwise_total = int(pcad_df["valid_pairwise_pcad_count"].sum())
    positive_pairwise_total = int(pcad_df["positive_pairwise_pcad_count"].sum())

    if (
        raise_if_valid_pairs_but_all_zero
        and valid_pairwise_total > 0
        and positive_pairwise_total == 0
    ):
        raise ValueError(
            f"All PCAD values are zero despite {valid_pairwise_total} valid "
            f"ego-neighbor pairs. Recording {recording_id:02d}, ego {ego_track_id}."
        )

    all_pairwise_nan = pcad_df[pcad_cols].isna().all(axis=1)

    pcad_df["critical_col"] = pd.Series(
        [pd.NA] * len(pcad_df),
        dtype="object"
    )

    pcad_df.loc[~all_pairwise_nan, "critical_col"] = (
        pcad_df.loc[~all_pairwise_nan, pcad_cols]
        .idxmax(axis=1)
    )

    pcad_df["critical_position"] = (
        pcad_df["critical_col"]
        .fillna("")
        .astype(str)
        .str.replace("pcad_", "", regex=False)
        .str.replace("Id", "", regex=False)
    )

    pcad_df.loc[
        pcad_df["pcad_max"].isna() | (pcad_df["pcad_max"] <= 0),
        "critical_position"
    ] = "No Risk"

    def get_critical_vehicle_id(row):
        if row["critical_position"] == "No Risk":
            return -1

        if pd.isna(row["critical_col"]):
            return -1

        neighbor_col = row["critical_col"].replace("pcad_", "")
        id_col = f"{neighbor_col}_trackId"

        if id_col not in row.index:
            return -1

        value = row[id_col]

        if pd.isna(value):
            return -1

        return int(value)

    pcad_df["critical_vehicle_id"] = pcad_df.apply(
        get_critical_vehicle_id,
        axis=1
    )

    max_pcad = pcad_df["pcad_max"].max()
    mean_pcad = pcad_df["pcad_max"].mean()
    mean_pairwise_pcad = pcad_df["pcad_mean"].mean()


    critical_position_frequency = (
        pcad_df["critical_position"]
        .value_counts(normalize=True)
        .mul(100)
        .to_dict()
    )

    risk_sum_by_position = {}

    for col in pcad_cols:
        position = (
            col.replace("pcad_", "")
               .replace("Id", "")
        )

        risk_sum_by_position[position] = float(
            pcad_df[col].sum(skipna=True)
        )

    total_risk = sum(risk_sum_by_position.values())

    if total_risk > 0:
        risk_contribution_by_position = {
            position: risk_sum / total_risk * 100
            for position, risk_sum in risk_sum_by_position.items()
        }
    else:
        risk_contribution_by_position = {
            position: 0.0
            for position in risk_sum_by_position.keys()
        }

    critical_sequence = pcad_df["critical_position"].copy()

    critical_sequence_no_risk = critical_sequence[
        critical_sequence != "No Risk"
    ]

    if len(critical_sequence_no_risk) == 0:
        critical_position_switch_count = 0
        dominant_critical_position = "No Critical Interaction"
    else:
        critical_position_switch_count = (
            critical_sequence_no_risk
            .ne(critical_sequence_no_risk.shift())
            .sum()
            - 1
        )

        critical_position_switch_count = max(
            int(critical_position_switch_count),
            0
        )

        dominant_critical_position = (
            critical_sequence_no_risk
            .value_counts()
            .idxmax()
        )

    average_speed = float(ego_df["lonVelocity"].mean())
    max_speed = float(ego_df["lonVelocity"].max())

    jerk_metrics = compute_jerk_metrics(
        ego_df=ego_df,
        frame_rate=frame_rate
    )

    summary = {
        "recording_id": recording_id,
        "merging_lanelet_id": merging_lanelet_id,
        "ego_track_id": ego_track_id,

        "frame_start": frame_start,
        "frame_end": frame_end,
        "num_frames": num_frames,
        "duration_s": duration_s,

        "max_pcad": max_pcad,
        "mean_pcad": mean_pcad,
        "mean_pairwise_pcad": mean_pairwise_pcad,

        "dominant_critical_position": dominant_critical_position,
        "critical_position_switch_count": critical_position_switch_count,

        "valid_pairwise_pcad_count": valid_pairwise_total,
        "positive_pairwise_pcad_count": positive_pairwise_total,

        "average_speed": average_speed,
        "max_speed": max_speed,

        "max_longitudinal_jerk": jerk_metrics["max_long_jerk"],
        "max_lateral_jerk": jerk_metrics["max_lat_jerk"],
        "rms_total_jerk": jerk_metrics["rms_total_jerk"],

        "status": "ok"
    }

    all_positions = [
        "lead",
        "rear",
        "leftLead",
        "leftRear",
        "leftAlongside",
        "rightLead",
        "rightRear",
        "rightAlongside",
        "No Risk"
    ]

    for position in all_positions:
        safe_name = (
            position
            .replace(" ", "_")
            .replace("-", "_")
        )

        summary[f"freq_{safe_name}_percent"] = (
            critical_position_frequency.get(position, 0.0)
        )

    for position in [
        "lead",
        "rear",
        "leftLead",
        "leftRear",
        "leftAlongside",
        "rightLead",
        "rightRear",
        "rightAlongside"
    ]:
        summary[f"risk_contribution_{position}_percent"] = (
            risk_contribution_by_position.get(position, 0.0)
        )

        summary[f"risk_sum_{position}"] = (
            risk_sum_by_position.get(position, 0.0)
        )

    if return_pcad_df:
        return summary, pcad_df

    return summary


# ============================================================
# Batch loop over exid_merging_vehicle_summary
# ============================================================
all_summary_rows = []
debug_pcad_dfs = {}

overall_start_time = time.time()

if save_raw_pcad and os.path.exists(raw_pcad_output_path):
    os.remove(raw_pcad_output_path)


for rec_idx, rec_row in merging_summary_df.iterrows():

    recording_id = int(rec_row["recording_id"])
    merging_lanelet_id = int(rec_row["merging_lanelet_id"])
    ego_track_ids = rec_row["merging_vehicle_ids"]

    print(
        f"\nProcessing recording {recording_id:02d} | "
        f"{len(ego_track_ids)} merging vehicles"
    )

    recording_start_time = time.time()

    raw_pcad_dfs_this_recording = []

    if recording_id not in tracks_file_map:
        raise FileNotFoundError(
            f"Missing tracks file for recording {recording_id:02d}."
        )

    if recording_id not in recording_meta_file_map:
        raise FileNotFoundError(
            f"Missing recordingMeta file for recording {recording_id:02d}."
        )

    tracks_path = tracks_file_map[recording_id]
    recording_meta_path = recording_meta_file_map[recording_id]

    try:
        tracks_df = pd.read_csv(
            tracks_path,
            usecols=required_cols,
            low_memory=False
        )

    except ValueError as e:
        header_cols = pd.read_csv(tracks_path, nrows=0).columns.tolist()
        missing_cols = [c for c in required_cols if c not in header_cols]

        raise ValueError(
            f"Recording {recording_id:02d} is missing required columns: "
            f"{missing_cols}"
        ) from e

    recording_meta_df = pd.read_csv(recording_meta_path)

    missing_cols = [c for c in required_cols if c not in tracks_df.columns]

    if missing_cols:
        raise ValueError(
            f"Recording {recording_id:02d} missing required columns: "
            f"{missing_cols}"
        )

    tracks_df = normalize_tracks_df_types(
        tracks_df=tracks_df,
        surrounding_cols=surrounding_cols
    )

    state_lookup = build_state_lookup(tracks_df)

    for i, ego_track_id in enumerate(ego_track_ids, start=1):

        ego_track_id = int(ego_track_id)

        print(
            f"  Processing vehicle {i}/{len(ego_track_ids)}: "
            f"ego_track_id = {ego_track_id}",
            flush=True
        )

        try:
            need_pcad_df = save_raw_pcad or save_debug_pcad

            if need_pcad_df:
                summary, pcad_df = compute_metrics_for_ego(
                    tracks_df=tracks_df,
                    state_lookup=state_lookup,
                    recording_meta_df=recording_meta_df,
                    recording_id=recording_id,
                    merging_lanelet_id=merging_lanelet_id,
                    ego_track_id=ego_track_id,
                    params=params,
                    return_pcad_df=True,
                    raise_if_valid_pairs_but_all_zero=raise_if_valid_pairs_but_all_zero
                )

                if save_debug_pcad:
                    debug_pcad_dfs[(recording_id, ego_track_id)] = pcad_df

                if save_raw_pcad and pcad_df is not None:
                    pcad_raw_df = pcad_df.copy()

                    frame_rate = (
                        float(recording_meta_df.loc[0, "frameRate"])
                        if "frameRate" in recording_meta_df.columns
                        else 25.0
                    )

                    frame_start = int(summary["frame_start"])

                    pcad_raw_df.insert(0, "ego_track_id", ego_track_id)
                    pcad_raw_df.insert(0, "merging_lanelet_id", merging_lanelet_id)
                    pcad_raw_df.insert(0, "recording_id", recording_id)

                    pcad_raw_df["time_s"] = (
                        pcad_raw_df["frame"] - frame_start
                    ) / frame_rate

                    raw_pcad_dfs_this_recording.append(pcad_raw_df)

            else:
                summary = compute_metrics_for_ego(
                    tracks_df=tracks_df,
                    state_lookup=state_lookup,
                    recording_meta_df=recording_meta_df,
                    recording_id=recording_id,
                    merging_lanelet_id=merging_lanelet_id,
                    ego_track_id=ego_track_id,
                    params=params,
                    return_pcad_df=False,
                    raise_if_valid_pairs_but_all_zero=raise_if_valid_pairs_but_all_zero
                )

        except Exception as e:
            if stop_on_error:
                raise RuntimeError(
                    f"Stopped because an error occurred while processing "
                    f"recording {recording_id:02d}, ego_track_id {ego_track_id}."
                ) from e

            summary = {
                "recording_id": recording_id,
                "merging_lanelet_id": merging_lanelet_id,
                "ego_track_id": ego_track_id,
                "status": f"error: {type(e).__name__}: {e}"
            }

        all_summary_rows.append(summary)

    print(
        f"  Finished {len(ego_track_ids)}/{len(ego_track_ids)} vehicles "
        f"in recording {recording_id:02d}"
    )

    if save_raw_pcad and len(raw_pcad_dfs_this_recording) > 0:

        raw_recording_df = pd.concat(
            raw_pcad_dfs_this_recording,
            ignore_index=True
        )

        write_header = not os.path.exists(raw_pcad_output_path)

        raw_recording_df.to_csv(
            raw_pcad_output_path,
            mode="a",
            header=write_header,
            index=False
        )

        print(
            f"  Saved raw PCAD rows for recording {recording_id:02d}: "
            f"{len(raw_recording_df)} rows"
        )

    recording_elapsed = time.time() - recording_start_time
    print(f"  Recording {recording_id:02d} elapsed time: {recording_elapsed:.1f} s")


# ============================================================
# Save final batch summary
# ============================================================
all_metrics_summary_df = pd.DataFrame(all_summary_rows)

all_metrics_summary_df.to_csv(output_path, index=False)

overall_elapsed = time.time() - overall_start_time

print("\nDone.")
print(f"Summary rows: {len(all_metrics_summary_df)}")
print(f"Summary saved to: {output_path}")

if save_raw_pcad:
    print(f"Raw PCAD saved to: {raw_pcad_output_path}")

print(f"Total elapsed time: {overall_elapsed:.1f} s")

display(all_metrics_summary_df.head())


# ============================================================
# Quick output check
# ============================================================
summary_check = pd.read_csv(output_path)

display(
    summary_check[
        [
            "recording_id",
            "ego_track_id",
            "max_pcad",
            "mean_pcad",
            "valid_pairwise_pcad_count",
            "positive_pairwise_pcad_count",
            "dominant_critical_position",
            "status"
        ]
    ]
)

if save_raw_pcad:
    raw_check = pd.read_csv(raw_pcad_output_path)

    print("\nRaw PCAD shape:", raw_check.shape)
    print("Raw PCAD file:", raw_pcad_output_path)

    pcad_cols_for_check = [
        col for col in raw_check.columns
        if col.startswith("pcad_")
        and col not in ["pcad_max", "pcad_mean"]
    ]

    print("\nMax value per PCAD column:")
    display(raw_check[pcad_cols_for_check + ["pcad_max"]].max())
